In [ ]:
!head ../bert/albert/CLUEdataset/tnews/train.json

In [ ]:
from fastai.vision.all import *
from torch.nn import *
from re import compile


def Zip(*a):
    return np.stack(a, 1)


def Flat(a):
    return a.ravel().tolist()


def Unzip(a, n):
    return np.array(a, "O").reshape(-1, n)


def Rand(a):
    return (np.random.rand(len(a)) * (a + 0.999)).astype(int)


def Slice(a, i, j):
    return a.split(Flat(np.diff(Zip(i, j).ravel(), prepend=0, append=len(a))))[1::2]


class Dat:
    m = {}

    def __init__(s, a, b=8):
        a = [loads(i) for i in open(f'../bert/albert/CLUEdataset/tnews/{a}.json')]
        s.y = torch.tensor([int(a["label"]) for a in a], device="cuda")
        if b:
            r = compile("[\u4e00-\u9fa5]")
            a = [[s.m.setdefault(a, len(s.m)) for a in r.findall(a["sentence"])] for a in a]
        else:
            a = [[s.m[a] for a in a["sentence"] if a in s.m] for a in a]
        s.x = torch.tensor([a for a in a for a in a], device="cuda")
        s.l = np.array([len(a) for a in a])
        s.L = np.r_[0, s.l[:-1]].cumsum()
        s.b = b

    def __len__(s):
        return len(s.y) // s.b if s.b else 1

    def __iter__(s):
        if s.b:
            for t in range(len(s)):
                t = np.random.choice(len(s.y), s.b, 0)
                t.sort()
                a = s.l[t]
                l = 1 + Rand(a - 1)
                i = s.L[t] + Rand(a - l)
                yield [torch.cat(Slice(s.x, i, i + l)), l], s.y[torch.tensor(t, device="cuda")]
        else:
            yield [s.x, s.l], s.y


class Embedding(Embedding):
    def forward(s, x):
        return super().forward(x[0]), x[1]


class BatchNorm1d(BatchNorm1d):
    def forward(s, x):
        return super().forward(x) if len(x) > 1 else x


class Red(Sequential):
    def forward(s, x):
        x, l = x
        while len(x) > len(l):
            t = Rand(l) & 65534
            l = Zip(t, l & 1, (l & 65534) - t)
            x = Unzip(x.split(Flat(l)), 3)
            l[:, ::2] >>= 1
            x[:, ::2] = Unzip(super().forward(torch.cat(Flat(x[:, ::2])).view(-1, x[0, 0].size(1) * 2)).split(Flat(l[:, ::2])), 2)
            x = torch.cat(Flat(x))
            l = l.sum(1)
        return x

In [ ]:
m = Learner(
    DataLoaders(Dat("train"), Dat("dev", 0)),
    Sequential(Embedding(len(Dat.m), 128), Red(Linear(256, 128), BatchNorm1d(128, track_running_stats=0), Swish()), Linear(128, 117)).cuda(),
    CrossEntropyLoss(),
    metrics=accuracy,
    cbs=ShowGraphCallback,
)
m.lr_find()

In [ ]:
m.fit_one_cycle(20, _[0], 9, 99, .01)

In [ ]:
m.fit_one_cycle(100, .001, 9, 99, .01)

In [ ]:
m.save('tnews')